In [1]:
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import LabelEncoder

# Load original CSV
df = pd.read_csv("urls_full_with_ip_domain_and_whois.csv")

# Step 1: Expand domain_properties
props = df['domain_properties'].apply(eval)  # Convert string to dict
df['domain_length'] = props.apply(lambda x: x.get('domain_length', -1))
df['has_digits'] = props.apply(lambda x: int(x.get('has_digits', False)))
df['has_hyphen'] = props.apply(lambda x: int(x.get('has_hyphen', False)))

# Step 2: Convert creation_date to domain_age_days
df['creation_date'] = pd.to_datetime(df['creation_date'], errors='coerce')
df['domain_age_days'] = (datetime.now() - df['creation_date']).dt.days
df['domain_age_days'] = df['domain_age_days'].fillna(-1)

# Step 3: Encode categorical features
for col in ['registrar', 'country', 'city']:
    df[col] = df[col].fillna('unknown')
    df[col] = LabelEncoder().fit_transform(df[col])

# Step 4: Encode label (type)
df['label'] = LabelEncoder().fit_transform(df['type'])  # phishing/benign/etc.

# Step 5: Drop unnecessary columns
df_cleaned = df.drop(columns=[
    'url', 'type', 'domain_properties', 'creation_date', 'error'
])

# Save cleaned data
df_cleaned.to_csv("cleaned_urls.csv", index=False)

print("✅ Cleaned data saved to 'cleaned_urls.csv'")


✅ Cleaned data saved to 'cleaned_urls.csv'
